# Phase 0: Colab GPU & Google Drive Verification
**Project:** Learn-a-World — JEPA World Model

This notebook verifies:
1. GPU acceleration (T4 GPU)
2. Google Drive mounting and persistence
3. Compressed `.npz` replay buffer chunk save & load

In [ ]:
# 1. Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU (change runtime to GPU if training heavy models)")

In [ ]:
# 2. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted successfully!")
except ImportError:
    print("Not running in Google Colab environment.")

In [ ]:
# 3. Test .npz Replay Buffer Chunk Save & Load
import os
import numpy as np

save_dir = "/content/drive/MyDrive/learn-a-world/test_checkpoints" if os.path.exists("/content/drive") else "./test_checkpoints"
os.makedirs(save_dir, exist_ok=True)
test_file = os.path.join(save_dir, "dummy_buffer_chunk.npz")

# Create synthetic Crafter-sized transitions (64x64x3 frames)
num_samples = 100
obs = np.random.randint(0, 256, size=(num_samples, 64, 64, 3), dtype=np.uint8)
actions = np.random.randint(0, 17, size=(num_samples,), dtype=np.int64)
rewards = np.random.randn(num_samples).astype(np.float32)
dones = np.zeros(num_samples, dtype=bool)

np.savez_compressed(test_file, obs=obs, actions=actions, rewards=rewards, dones=dones)
print(f"Saved dummy chunk ({os.path.getsize(test_file) / 1024:.2f} KB) to {test_file}")

# Reload and verify data integrity
loaded = np.load(test_file)
assert np.array_equal(loaded["obs"], obs), "Obs mismatch!"
assert np.array_equal(loaded["actions"], actions), "Actions mismatch!"
assert np.allclose(loaded["rewards"], rewards), "Rewards mismatch!"
print("Gate Test Passed: Google Drive buffer persistence verified!")